### Setup Data loader

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader

from agent.components import RASK

raw_df = pd.read_csv("../statics/metrics_20_0.csv")
converted_df = RASK.preprocess_data(raw_df)


# train_loader = link_service_data(converted_df)

def prepare_chained_data(df: pd.DataFrame, test_size=0.2) -> DataLoader:
    # Split the interleaved rows
    df_qr = df.iloc[0::3].reset_index(drop=True)
    df_cv = df.iloc[1::3].reset_index(drop=True)
    df_pc = df.iloc[2::3].reset_index(drop=True)

    # Feature selection
    X_qr = df_qr[['cores', 'data_quality']].values
    X_cv = df_cv[['cores', 'data_quality', 'model_size']].values
    X_pc = df_pc[['cores', 'data_quality']].values
    X_combined = np.hstack([X_qr, X_cv, X_pc])

    # TODO: Actually they would have to depend on each other, like one overall latency?
    # Targets
    Y_vals = np.column_stack([df_qr['throughput'].values,
                              df_cv['throughput'].values,
                              df_pc['throughput'].values])

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_x.fit_transform(X_combined)
    Y_scaled = scaler_y.fit_transform(Y_vals)

    # Train/Test Split
    x_train, x_test, y_train, y_test = train_test_split(X_scaled, Y_scaled, test_size=test_size)

    # To Tensors
    t_x_train = torch.tensor(x_train, dtype=torch.float32)
    t_x_test = torch.tensor(x_test, dtype=torch.float32)
    t_y_train = torch.tensor(y_train, dtype=torch.float32)
    t_y_test = torch.tensor(y_test, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(t_x_train, t_y_train), batch_size=64, shuffle=True)

    return train_loader, t_x_test, t_y_test, scaler_y

In [ ]:
# import gpytorch
#
# test_dataset = TensorDataset(test_x, test_y)
# test_loader = DataLoader(test_dataset, batch_size=1024)
#
# model.eval()
# predictive_means, predictive_variances, test_lls = model.predict(test_loader)
#
# rmse = torch.mean(torch.pow(predictive_means.mean(0) - test_y, 2)).sqrt()
# print(f"RMSE: {rmse.item()}, NLL: {-test_lls.mean().item()}")

### Setup the Structure


In [ ]:
import torch
import gpytorch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# --- 1. MODEL DEFINITION ---

class ServiceGP(gpytorch.models.ApproximateGP):
    """A standard Variational GP for an individual service."""

    def __init__(self, input_dims, num_inducing=128):
        inducing_points = torch.randn(num_inducing, input_dims)
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(num_inducing)
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        super().__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=input_dims)
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class ServiceChain(torch.nn.Module):
    """Connects 3 GPs into a chain where outputs of one become inputs to the next."""

    def __init__(self, qr_idx, cv_idx, pc_idx):
        super().__init__()
        self.qr_idx, self.cv_idx, self.pc_idx = qr_idx, cv_idx, pc_idx

        # input_dims = raw_features + 1 (from previous service sample)
        self.qr_gp = ServiceGP(input_dims=len(qr_idx))
        self.cv_gp = ServiceGP(input_dims=len(cv_idx) + 1)
        self.pc_gp = ServiceGP(input_dims=len(pc_idx) + 1)

        self.qr_likelihood = gpytorch.likelihoods.GaussianLikelihood()
        self.cv_likelihood = gpytorch.likelihoods.GaussianLikelihood()
        self.pc_likelihood = gpytorch.likelihoods.GaussianLikelihood()

    def forward(self, x):
        # Service 1: QR
        qr_dist = self.qr_gp(x[:, self.qr_idx])
        qr_samples = qr_dist.rsample()  # Sample to propagate uncertainty

        # Service 2: CV (Features + QR Latent)
        cv_input = torch.cat([x[:, self.cv_idx], qr_samples.unsqueeze(-1)], dim=-1)
        cv_dist = self.cv_gp(cv_input)
        cv_samples = cv_dist.rsample()

        # Service 3: PC (Features + CV Latent)
        pc_input = torch.cat([x[:, self.pc_idx], cv_samples.unsqueeze(-1)], dim=-1)
        pc_dist = self.pc_gp(pc_input)

        return qr_dist, cv_dist, pc_dist

### Do the training of the GP

In [ ]:
# Load and Process (Assuming converted_df exists from your RASK.preprocess)
train_loader, test_x, test_y, y_scaler = prepare_chained_data(converted_df)

model = ServiceChain(qr_idx=[0, 1], cv_idx=[2, 3, 4], pc_idx=[5, 6])
if torch.cuda.is_available():
    model, test_x, test_y = model.cuda(), test_x.cuda(), test_y.cuda()

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
mll_qr = gpytorch.mlls.VariationalELBO(model.qr_likelihood, model.qr_gp, num_data=len(train_loader.dataset))
mll_cv = gpytorch.mlls.VariationalELBO(model.cv_likelihood, model.cv_gp, num_data=len(train_loader.dataset))
mll_pc = gpytorch.mlls.VariationalELBO(model.pc_likelihood, model.pc_gp, num_data=len(train_loader.dataset))

model.train()
for epoch in range(200):
    with gpytorch.settings.cholesky_jitter(1e-3):
        for x_batch, y_batch in train_loader:
            if torch.cuda.is_available(): x_batch, y_batch = x_batch.cuda(), y_batch.cuda()

            optimizer.zero_grad()
            qr_d, cv_d, pc_d = model(x_batch)

            # Loss is sum of ELBOs for all components
            loss = -mll_qr(qr_d, y_batch[:, 0]) - mll_cv(cv_d, y_batch[:, 1]) - mll_pc(pc_d, y_batch[:, 2])

            loss.backward()
            optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


### Do some sort of inference

In [ ]:
model.eval()
with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
    # Predict distributions
    qr_d, cv_d, pc_d = model(test_x)

    # Generate 1000 Monte Carlo samples for the whole chain
    num_mc = 1000
    qr_samples = qr_d.sample(torch.Size([num_mc]))  # (1000, N)
    cv_samples = cv_d.sample(torch.Size([num_mc]))
    pc_samples = pc_d.sample(torch.Size([num_mc]))

    # Combine into a single sample tensor (1000, N, 3) to unscale
    combined_samples = torch.stack([qr_samples, cv_samples, pc_samples], dim=-1).cpu().numpy()

    # Unscale samples to real-world units
    # Reshape to 2D for inverse_transform, then back to 3D
    orig_shape = combined_samples.shape
    unscaled_samples = y_scaler.inverse_transform(combined_samples.reshape(-1, 3)).reshape(orig_shape)

    # EXAMPLE SLO: Chain-wide latency (Sum) < 1500ms
    # unscaled_samples[:, :, 0] is QR, [:, :, 1] is CV, etc.
    total_metric = unscaled_samples.sum(axis=-1)

    # TODO: If your SLO is about Throughput, use total_metric = unscaled_samples.min(axis=-1).
    threshold = 1500.0
    slo_fulfilled = (total_metric < threshold).mean(axis=0)  # Mean across samples = Probability

print("\n--- SLO Analysis ---")
print(f"Average probability of chain fulfillment: {slo_fulfilled.mean():.2%}")
print(f"Probabilities for first 5 test cases: {slo_fulfilled[:5]}")



In [10]:

# --- MODEL VALIDATION ---
# Get the mean prediction for each service (mean across Monte Carlo samples)
# unscaled_samples shape is (1000, N_test, 3)
predicted_means = unscaled_samples.mean(axis=0)

# Unscale the ground truth test_y to compare in real units
actual_y = y_scaler.inverse_transform(test_y.cpu().numpy())

# Calculate Error for each service
for i, name in enumerate(['QR', 'CV', 'PC']):
    rmse = np.sqrt(np.mean((predicted_means[:, i] - actual_y[:, i]) ** 2))
    print(f"{name} Service RMSE: {rmse:.4f}")

QR Service RMSE: 2.1386
CV Service RMSE: 0.4901
PC Service RMSE: 1.1520
